In [ ]:
import pandas as pd
import numpy as np



In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.losses import BinaryCrossentropy

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

In [3]:
data = pd.read_csv("Final.csv")
data.head()


,Crop,Stage,Disease Name,Pathogen Type,Favorable Weather Conditions,Symptoms at This Stage,Why This Stage is Vulnerable,Preventive Measures,Monitoring Indicators,NDVI,...,RVI,VARI,SOC,AVI,BSI,SI,VSSI,Crop_enc,Stage_enc,Disease_enc
0,corn,seed treatment & sowing,seed rot & seedling blight (pythium spp.),Fungus (Oomycete),"Cool, wet soils; soil temperature < 15–18°C; h...","Seeds fail to germinate, soft rotten seeds, we...",Seeds are in prolonged contact with moist soil...,Use treated seeds; ensure well-drained soil; a...,Soil moisture levels >70%; NDVI shows no early...,2,...,2,2,2,2,2,2,2,2,22,183
1,corn,seed treatment & sowing,seedling blight (fusarium spp.),Fungus,Warm + moist soils; fluctuating temperatures; ...,"Poor emergence, reddish/pink discoloration on ...",Fusarium survives in soil and infects slow-ger...,Use fungicide-treated seeds; crop rotation; av...,Soil temp fluctuations; erratic NDVI rise; his...,2,...,2,2,2,2,2,2,2,2,22,187
2,corn,seed treatment & sowing,seedling anthracnose,Fungus,Warm and humid; rainfall during sowing,"Brown to black lesions on emerging coleoptile,...",The pathogen infects shoots as soon as seedlin...,Certified seeds; treat seeds with broad-spectr...,High humidity (>85%); rainfall immediately aft...,3,...,2,2,2,2,2,2,2,2,22,185
3,corn,seed treatment & sowing,common smut,Fungus,Dry conditions followed by brief moisture; war...,Usually asymptomatic at sowing; may show sligh...,Infection occurs through young meristematic ti...,Use resistant hybrids; seed treatment; avoid m...,Soil injury zones; fluctuating humidity; NDVI ...,3,...,4,4,4,4,2,2,2,2,22,56
4,corn,seed treatment & sowing,seed corn maggot,Pest (Insect Larvae),"Cool, wet soils; heavy organic matter; recentl...","Seeds hollowed out, missing seeds, weak emergence",Slow germination due to low GDD (10) makes see...,Avoid fresh manure before sowing; use insectic...,Pest trap counts; soil organic matter; NDVI ga...,2,...,2,2,2,2,2,2,2,2,22,181


In [4]:
FEATURE_COLUMNS = [
    "Crop_enc",
    "Stage_enc",
    "Pathogen Type",
    "NDVI","EVI","SAVI","NDRE","LAI",
    "NDWI","NDMI",
    "RSM","RVI","VARI",
    "SOC","AVI","BSI","SI","VSSI"
]

TARGET_COLUMN = "Disease Name"

data = data[FEATURE_COLUMNS + [TARGET_COLUMN]]
data.head()


,Crop_enc,Stage_enc,Pathogen Type,NDVI,EVI,SAVI,NDRE,LAI,NDWI,NDMI,RSM,RVI,VARI,SOC,AVI,BSI,SI,VSSI,Disease Name
0,2,22,Fungus (Oomycete),2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,seed rot & seedling blight (pythium spp.)
1,2,22,Fungus,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,seedling blight (fusarium spp.)
2,2,22,Fungus,3,3,3,3,3,3,3,3,2,2,2,2,2,2,2,seedling anthracnose
3,2,22,Fungus,3,3,3,3,3,3,3,3,4,4,4,4,2,2,2,common smut
4,2,22,Pest (Insect Larvae),2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,seed corn maggot


In [5]:
pathogen_le = LabelEncoder()
data["Pathogen_enc"] = pathogen_le.fit_transform(data["Pathogen Type"])

data.drop(columns=["Pathogen Type"], inplace=True)
data.head()


,Crop_enc,Stage_enc,NDVI,EVI,SAVI,NDRE,LAI,NDWI,NDMI,RSM,RVI,VARI,SOC,AVI,BSI,SI,VSSI,Disease Name,Pathogen_enc
0,2,22,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,seed rot & seedling blight (pythium spp.),20
1,2,22,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,seedling blight (fusarium spp.),8
2,2,22,3,3,3,3,3,3,3,3,2,2,2,2,2,2,2,seedling anthracnose,8
3,2,22,3,3,3,3,3,3,3,3,4,4,4,4,2,2,2,common smut,8
4,2,22,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,seed corn maggot,44


In [6]:
X = data.drop(columns=[TARGET_COLUMN]).values
X.shape


(444, 18)

In [7]:
group_cols = list(data.drop(columns=[TARGET_COLUMN]).columns)

grouped = (
    data
    .groupby(group_cols)[TARGET_COLUMN]
    .apply(list)
    .reset_index()
)

grouped.head()


,Crop_enc,Stage_enc,NDVI,EVI,SAVI,NDRE,LAI,NDWI,NDMI,RSM,RVI,VARI,SOC,AVI,BSI,SI,VSSI,Pathogen_enc,Disease Name
0,0,8,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,8,[fusarium wilt]
1,0,8,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,55,[sterility mosaic disease]
2,0,8,3,3,3,3,3,3,3,3,2,2,2,2,2,2,2,8,[powdery mildew]
3,0,8,3,3,3,3,3,3,3,3,4,4,4,4,4,4,4,8,[phytophthora blight]
4,0,8,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,8,[alternaria blight]


In [8]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(grouped[TARGET_COLUMN])

X = grouped.drop(columns=[TARGET_COLUMN]).values

X.shape, Y.shape


((335, 18), (335, 235))

In [9]:
def build_tabnet_like(input_dim, output_dim):
    inputs = layers.Input(shape=(input_dim,))

    # Feature transformer
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)

    # Attentive step
    attn = layers.Dense(input_dim, activation="sigmoid")(x)
    x = layers.Multiply()([inputs, attn])

    # Decision layers
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(
        output_dim,
        activation="sigmoid"   # 🔥 multi-label
    )(x)

    return models.Model(inputs, outputs)


In [10]:
model = build_tabnet_like(
    input_dim=X.shape[1],
    output_dim=Y.shape[1]
)

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=BinaryCrossentropy()
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 18)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      2,432 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 18)        │      2,322 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 18)        │          0 │ input_layer[0][0… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │      2,432 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 235)       │     30,315 │ batch_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,525 (150.49 KB)

 Trainable params: 38,013 (148.49 KB)

 Non-trainable params: 512 (2.00 KB)

In [14]:
history = model.fit(
    X,
    Y,
    epochs=80,
    batch_size=16,
    verbose=1
)


Epoch 1/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0093 
Epoch 2/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0093 
Epoch 3/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0088 
Epoch 4/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0090 
Epoch 5/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0096 
Epoch 6/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0092 
Epoch 7/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0088 
Epoch 8/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0088 
Epoch 9/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0096 
Epoch 10/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0090 
Epoch 11/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0097 
Epoch 12/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0087 
Epoch 13/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0088 
Epoch 14/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0090 
Epoch 15/80
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0094 
Epoc

In [15]:
def predict_diseases(input_row, threshold=0.4):
    probs = model.predict(input_row.reshape(1, -1))[0]
    return mlb.classes_[probs >= threshold].tolist()


In [16]:
for i in range(5):
    predicted = predict_diseases(X[i])
    actual = grouped.iloc[i][TARGET_COLUMN]

    print("Actual:", set(actual))
    print("Predicted:", set(predicted))
    print("-" * 50)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Actual: {'fusarium wilt'}
Predicted: {'fusarium wilt'}
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Actual: {'sterility mosaic disease'}
Predicted: {'sterility mosaic disease'}
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Actual: {'powdery mildew'}
Predicted: {'alternaria blight', 'powdery mildew'}
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Actual: {'phytophthora blight'}
Predicted: {'phytophthora blight'}
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Actual: {'alternaria blight'}
Predicted: {'alternaria blight'}
--------------------------------------------------


In [22]:
raw_data = pd.read_csv("Final.csv")

raw_data = raw_data[["Crop", "Stage", "Disease Name"]]
raw_data.head()


,Crop,Stage,Disease Name
0,corn,seed treatment & sowing,seed rot & seedling blight (pythium spp.)
1,corn,seed treatment & sowing,seedling blight (fusarium spp.)
2,corn,seed treatment & sowing,seedling anthracnose
3,corn,seed treatment & sowing,common smut
4,corn,seed treatment & sowing,seed corn maggot


In [23]:
# disease name -> index (from trained mlb)
disease_to_idx = {d: i for i, d in enumerate(mlb.classes_)}

stage_disease_idx_map = (
    raw_data
    .groupby(["Crop", "Stage"])["Disease Name"]
    .apply(lambda x: sorted(set(disease_to_idx[d] for d in x)))
    .to_dict()
)


In [24]:
# quick sanity check
list(stage_disease_idx_map.items())[:3]


[(('chickpea', 'emergence'), [21, 49, 76, 84, 102, 154, 163, 169]),
 (('chickpea', 'flowering'), [2, 17, 20, 44, 84, 103, 150, 169, 176]),
 (('chickpea', 'germination'), [23, 49, 76, 84, 102, 154, 163, 169])]

In [25]:
def predict_stage_disease_risk(
    input_vector,
    crop,
    stage,
    normalize=True
):
    probs = model.predict(input_vector.reshape(1, -1))[0]

    valid_indices = stage_disease_idx_map.get((crop, stage), [])

    disease_risks = {
        mlb.classes_[i]: probs[i]
        for i in valid_indices
    }

    if normalize and disease_risks:
        total = sum(disease_risks.values())
        disease_risks = {
            d: round((p / total) * 100, 2)
            for d, p in disease_risks.items()
        }
    else:
        disease_risks = {
            d: round(p * 100, 2)
            for d, p in disease_risks.items()
        }

    return disease_risks


In [26]:
crop = raw_data.iloc[0]["Crop"]
stage = raw_data.iloc[0]["Stage"]

print("Crop:", crop)
print("Stage:", stage)

input_vector = X[0]  # same row features
print(predict_stage_disease_risk(input_vector, crop, stage))


Crop: corn
Stage: seed treatment & sowing
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
{'common smut': np.float32(0.96), 'seed corn maggot': np.float32(4.72), 'seed rot & seedling blight (pythium spp.)': np.float32(8.41), 'seedling anthracnose': np.float32(1.4), 'seedling blight (fusarium spp.)': np.float32(6.17), 'stewart’s wilt': np.float32(78.34)}


In [27]:
INDEX_COLUMNS = [
    "NDVI","EVI","SAVI","NDRE","LAI",
    "NDWI","NDMI",
    "RSM","RVI","VARI",
    "SOC","AVI","BSI","SI","VSSI"
]


In [28]:
def build_input_vector_with_indices(
    crop_enc,
    stage_enc,
    pathogen_type,
    indices_dict
):
    pathogen_enc = encode_pathogen_safe(pathogen_type)

    feature_vector = [
        crop_enc,
        stage_enc,
        pathogen_enc,
    ] + [indices_dict[col] for col in INDEX_COLUMNS]

    return (
        np.array(feature_vector, dtype=np.float32),
        {col: indices_dict[col] for col in INDEX_COLUMNS}
    )


In [29]:
def predict_stage_disease_risk_with_indices(
    crop,
    stage,
    crop_enc,
    stage_enc,
    pathogen_type,
    indices_dict,
    normalize=True
):
    # build input
    input_vector, used_indices = build_input_vector_with_indices(
        crop_enc,
        stage_enc,
        pathogen_type,
        indices_dict
    )

    # model probabilities
    probs = model.predict(input_vector.reshape(1, -1))[0]

    # diseases valid for this crop-stage
    valid_indices = stage_disease_idx_map.get((crop, stage), [])

    disease_risks = {
        mlb.classes_[i]: probs[i]
        for i in valid_indices
    }

    # normalize to percentage
    if normalize and disease_risks:
        total = sum(disease_risks.values())
        disease_risks = {
            d: round((p / total) * 100, 2)
            for d, p in disease_risks.items()
        }
    else:
        disease_risks = {
            d: round(p * 100, 2)
            for d, p in disease_risks.items()
        }

    return {
        "crop": crop,
        "stage": stage,
        "indices_used": used_indices,
        "disease_risk": disease_risks
    }


In [32]:
def encode_pathogen_safe(value):
    """
    Safely encodes pathogen type (case-insensitive).
    Throws a clear error if value is invalid.
    """
    value = value.strip().lower()

    classes = [c.lower() for c in pathogen_le.classes_]

    if value not in classes:
        raise ValueError(
            f"Invalid pathogen type: '{value}'. "
            f"Allowed values: {pathogen_le.classes_.tolist()}"
        )

    # map back to original encoder class
    original_class = pathogen_le.classes_[classes.index(value)]
    return pathogen_le.transform([original_class])[0]


In [34]:
print("Looking for key:", ("chickpea", "vegetative"))
print("Available keys sample:", list(stage_disease_idx_map.keys())[:10])


Looking for key: ('chickpea', 'vegetative')
Available keys sample: [('chickpea', 'emergence'), ('chickpea', 'flowering'), ('chickpea', 'germination'), ('chickpea', 'harvest'), ('chickpea', 'maturity'), ('chickpea', 'pod formation (reproductive stage)'), ('chickpea', 'post-harvest'), ('chickpea', 'seed treatment & sowing'), ('chickpea', 'vegetative growth'), ('corn', 'blister (r2)')]


In [35]:
STAGE_ALIAS_MAP = {
    # vegetative
    "vegetative": "vegetative growth",
    "vegetative stage": "vegetative growth",
    "veg": "vegetative growth",

    # reproductive
    "reproductive": "pod formation (reproductive stage)",
    "pod formation": "pod formation (reproductive stage)",

    # sowing
    "sowing": "seed treatment & sowing",
    "seed sowing": "seed treatment & sowing",
    "germination": "germination",

    # others (optional, extend later)
    "flowering": "flowering",
    "maturity": "maturity",
    "harvest": "harvest",
    "post harvest": "post-harvest",
    "emergence": "emergence",
}


In [36]:
def resolve_stage(stage_input):
    stage_input = stage_input.strip().lower()

    if stage_input in STAGE_ALIAS_MAP:
        return STAGE_ALIAS_MAP[stage_input]

    return stage_input


In [37]:
def predict_stage_disease_risk_with_indices(
    crop,
    stage,
    crop_enc,
    stage_enc,
    pathogen_type,
    indices_dict,
    normalize=True
):
    crop_key = crop.strip().lower()
    stage_key = resolve_stage(stage)

    input_vector, used_indices = build_input_vector_with_indices(
        crop_enc,
        stage_enc,
        pathogen_type,
        indices_dict
    )

    probs = model.predict(input_vector.reshape(1, -1))[0]

    valid_indices = stage_disease_idx_map.get((crop_key, stage_key), [])

    disease_risks = {
        mlb.classes_[i]: probs[i]
        for i in valid_indices
    }

    if normalize and disease_risks:
        total = sum(disease_risks.values())
        disease_risks = {
            d: round((p / total) * 100, 2)
            for d, p in disease_risks.items()
        }

    return {
        "crop": crop,
        "stage_input": stage,
        "stage_resolved": stage_key,
        "indices_used": used_indices,
        "disease_risk": disease_risks
    }


In [38]:
result = predict_stage_disease_risk_with_indices(
    crop="chickpea",
    stage="vegetative",   # 👈 user-friendly
    crop_enc=3,
    stage_enc=1,
    pathogen_type="Fungus",
    indices_dict={
        "NDVI": 2, "EVI": 2, "SAVI": 2, "NDRE": 3, "LAI": 2,
        "NDWI": 3, "NDMI": 3,
        "RSM": 2, "RVI": 2, "VARI": 2,
        "SOC": 2, "AVI": 3, "BSI": 2, "SI": 2, "VSSI": 3
    }
)

result


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


{'crop': 'chickpea',
 'stage_input': 'vegetative',
 'stage_resolved': 'vegetative growth',
 'indices_used': {'NDVI': 2,
  'EVI': 2,
  'SAVI': 2,
  'NDRE': 3,
  'LAI': 2,
  'NDWI': 3,
  'NDMI': 3,
  'RSM': 2,
  'RVI': 2,
  'VARI': 2,
  'SOC': 2,
  'AVI': 3,
  'BSI': 2,
  'SI': 2,
  'VSSI': 3},
 'disease_risk': {'alternaria blight (alternaria alternata)': np.float32(0.0),
  'ascochyta blight (ascochyta rabiei)': np.float32(99.93),
  'botrytis gray mold (botrytis cinerea)': np.float32(0.0),
  'collar rot (sclerotium rolfsii)': np.float32(0.0),
  'dry root rot (macrophomina phaseolina)': np.float32(0.0),
  'fusarium wilt (fusarium oxysporum f. sp. ciceri)': np.float32(0.0),
  'helicoverpa (early feeding stage)': np.float32(0.0),
  'root-knot nematode (meloidogyne spp.)': np.float32(0.07)}}

In [17]:
def build_input_vector(
    crop_enc,
    stage_enc,
    pathogen_type,
    indices_dict
):
    """
    crop_enc: int (already encoded)
    stage_enc: int (already encoded)
    pathogen_type: string
    indices_dict: dict of index_name -> value
    """

    pathogen_enc = pathogen_le.transform([pathogen_type])[0]

    feature_vector = [
        crop_enc,
        stage_enc,
        pathogen_enc,
        indices_dict["NDVI"],
        indices_dict["EVI"],
        indices_dict["SAVI"],
        indices_dict["NDRE"],
        indices_dict["LAI"],
        indices_dict["NDWI"],
        indices_dict["NDMI"],
        indices_dict["RSM"],
        indices_dict["RVI"],
        indices_dict["VARI"],
        indices_dict["SOC"],
        indices_dict["AVI"],
        indices_dict["BSI"],
        indices_dict["SI"],
        indices_dict["VSSI"],
    ]

    return np.array(feature_vector, dtype=np.float32)


In [19]:
# Example input (you can change values)
input_vector = build_input_vector(
    crop_enc=3,              # example: chickpea
    stage_enc=1,             # example: vegetative
    pathogen_type="Fungus",
    indices_dict={
        "NDVI": 2,
        "EVI": 2,
        "SAVI": 2,
        "NDRE": 3,
        "LAI": 2,
        "NDWI": 3,
        "NDMI": 3,
        "RSM": 2,
        "RVI": 2,
        "VARI": 2,
        "SOC": 2,
        "AVI": 3,
        "BSI": 2,
        "SI": 2,
        "VSSI": 3,
    }
)

input_vector.shape


(18,)

In [20]:
pathogen_le.classes_


array(['Abiotic', 'Bacteria', 'Bacteria (Pantoea stewartii)',
       'Bacteria (Xanthomonas citri pv. malvacearum)', 'Bacteria + Fungi',
       'Bacteria / Yeast', 'Fungal', 'Fungal Toxin', 'Fungus',
       'Fungus (Aspergillus flavus)', 'Fungus (Bipolaris maydis)',
       'Fungus (Cercospora zeae-maydis)',
       'Fungus (Colletotrichum graminicola)',
       'Fungus (Exserohilum turcicum)', 'Fungus (Fusarium graminearum)',
       'Fungus (Fusarium spp., Pythium spp.)',
       'Fungus (Fusarium verticillioides)',
       'Fungus (Fusarium, Rhizoctonia)',
       'Fungus (Macrophomina phaseolina)', 'Fungus (Macrophomina)',
       'Fungus (Oomycete)', 'Fungus (Phyllachora maydis)',
       'Fungus (Puccinia polysora)', 'Fungus (Puccinia sorghi)',
       'Fungus (Pythium, Rhizoctonia solani)',
       'Fungus (Pythium, Rhizoctonia)',
       'Fungus (Rhizoctonia, Aspergillus, Fusarium)',
       'Fungus (Rhizoctonia, Pythium, Phytophthora)',
       'Fungus (Stenocarpella maydis)', 'Fungus (Usti